In [ ]:
# Cell 1 — Imports
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import joblib
import json
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from utils.preprocessing import (
    normalize_education, normalize_specialization,
    normalize_skills_list, normalize_career, split_semicolon_field
)

In [ ]:
# Cell 2 — Load and normalize data
df = pd.read_csv("../data/processed/career_training.csv")
print(f"Shape: {df.shape}")
print(df.dtypes)

# Normalize columns
df["Education Level"] = df["Education Level"].apply(normalize_education)
df["Specialization"] = df["Specialization"].apply(normalize_specialization)
df["Recommended Career"] = df["Recommended Career"].apply(normalize_career)

# Parse Skills: split on ';' -> list
df["Skills"] = df["Skills"].apply(lambda x: split_semicolon_field(str(x)))
df["Skills"] = df["Skills"].apply(normalize_skills_list)

print(df.head())
print(f"Unique careers: {df['Recommended Career'].nunique()}")

In [ ]:
# Cell 3 — Encode target and build preprocessor

# Encode target
le = LabelEncoder()
y = le.fit_transform(df["Recommended Career"])
joblib.dump(le, "../models/label_encoder.joblib")
print(f"Classes: {list(le.classes_)}")
print(f"y shape: {y.shape}")

# Feature matrix
X = df[["Education Level", "Specialization", "Skills"]].copy()

# Build ColumnTransformer:
# - MultiLabelBinarizer for Skills (list column)
# - OrdinalEncoder for Education Level and Specialization
from sklearn.pipeline import make_pipeline

# MultiLabelBinarizer doesn't work natively in ColumnTransformer,
# so we wrap it in a custom transformer or use FunctionTransformer + MLB
from sklearn.base import BaseEstimator, TransformerMixin

class MultiLabelBinarizerTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.mlb = MultiLabelBinarizer()
    
    def fit(self, X, y=None):
        self.mlb.fit(X)
        return self
    
    def transform(self, X, y=None):
        return self.mlb.transform(X)

preprocessor = ColumnTransformer(transformers=[
    ("skills", MultiLabelBinarizerTransformer(), "Skills"),
    ("edu", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), ["Education Level"]),
    ("spec", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), ["Specialization"]),
])

print("Preprocessor built.")

In [ ]:
# Cell 4 — Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")

In [ ]:
# Cell 5 — Define models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, multi_class="multinomial", random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(kernel="rbf", probability=True, random_state=42),
}

In [ ]:
# Cell 6 — Train and evaluate each model
results = {}

for model_name, model in models.items():
    print(f"Training {model_name}...")
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])
    # Fit only on training data — no test data leakage
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    
    results[model_name] = {
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "f1_macro": float(f1_score(y_test, y_pred, average="macro")),
        "precision": float(precision_score(y_test, y_pred, average="macro", zero_division=0)),
        "recall": float(recall_score(y_test, y_pred, average="macro", zero_division=0)),
    }
    
    print(f"  {model_name}: acc={results[model_name]['accuracy']:.4f}, "
          f"f1={results[model_name]['f1_macro']:.4f}")

print("\nAll models trained.")

In [ ]:
# Cell 7 — Select best model and save artifacts

# Select model with highest f1_macro
best_name = max(results, key=lambda k: results[k]["f1_macro"])
print(f"Best model: {best_name} (f1_macro={results[best_name]['f1_macro']:.4f})")

# Refit best pipeline on training data
best_model = models[best_name]
best_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", best_model)
])
best_pipeline.fit(X_train, y_train)

# Save best pipeline
joblib.dump(best_pipeline, "../models/career_model.joblib")
print("Saved: models/career_model.joblib")

# Save preprocessor separately
joblib.dump(best_pipeline.named_steps["preprocessor"], "../models/preprocessor.joblib")
print("Saved: models/preprocessor.joblib")

# Save evaluation results
with open("../data/processed/model_evaluation.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved: data/processed/model_evaluation.json")

print("\nEvaluation Summary:")
for name, metrics in results.items():
    print(f"  {name}: f1_macro={metrics['f1_macro']:.4f}, accuracy={metrics['accuracy']:.4f}")

In [ ]:
# Cell 8 — Verify artifacts exist
import os

artifacts = [
    "../models/career_model.joblib",
    "../models/preprocessor.joblib",
    "../models/label_encoder.joblib",
]

for path in artifacts:
    exists = os.path.exists(path)
    print(f"{'✓' if exists else '✗'} {path}")

# Quick sanity check: reload and predict
model = joblib.load("../models/career_model.joblib")
le_check = joblib.load("../models/label_encoder.joblib")
test_input = pd.DataFrame([{
    "Education Level": "Bachelor",
    "Specialization": "Computer Science",
    "Skills": ["Python", "SQL", "Machine Learning"]
}])
proba = model.predict_proba(test_input)[0]
top_idx = proba.argsort()[::-1][:5]
print("\nSanity check — top 5 predictions:")
for i, idx in enumerate(top_idx):
    print(f"  {i+1}. {le_check.classes_[idx]}: {proba[idx]:.4f}")